In [1]:
!pip -q install awscli nilearn nibabel pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but you have docutils 0.19 which is incompatible.


In [2]:
import os
import numpy as np
import pandas as pd
from nilearn import datasets
from nilearn.maskers import NiftiLabelsMasker
from nilearn.connectome import ConnectivityMeasure

In [3]:
os.makedirs("/content/ds006072_work/raw", exist_ok=True)
os.makedirs("/content/ds006072_work/matrices", exist_ok=True)

In [4]:
atlas = datasets.fetch_atlas_schaefer_2018(
    n_rois=100,
    yeo_networks=7
)

masker = NiftiLabelsMasker(
    labels_img=atlas.maps,
    standardize=True,
    verbose=0
)

corr = ConnectivityMeasure(kind="correlation")

[fetch_atlas_schaefer_2018] Added README.md to /root/nilearn_data

[fetch_atlas_schaefer_2018] Dataset created in /root/nilearn_data/schaefer_2018

[fetch_atlas_schaefer_2018] Downloading data from 
https://raw.githubusercontent.com/ThomasYeoLab/CBIG/v0.14.3-Update_Yeo2011_Schaefer2018_labelname/stable_projects/b
rain_parcellation/Schaefer2018_LocalGlobal/Parcellations/MNI/Schaefer2018_100Parcels_7Networks_order.txt ...

[fetch_atlas_schaefer_2018]  ...done. (0 seconds, 0 min)

[fetch_atlas_schaefer_2018] Downloading data from 
https://raw.githubusercontent.com/ThomasYeoLab/CBIG/v0.14.3-Update_Yeo2011_Schaefer2018_labelname/stable_projects/b
rain_parcellation/Schaefer2018_LocalGlobal/Parcellations/MNI/Schaefer2018_100Parcels_7Networks_order_FSLMNI152_1mm.
nii.gz ...

[fetch_atlas_schaefer_2018]  ...done. (0 seconds, 0 min)

In [5]:
subjects = ["sub-P1", "sub-P2", "sub-P3"]
subjects

['sub-P1', 'sub-P2', 'sub-P3']

In [6]:
def process_subject(sub):
    remote_file = f"{sub}/ses-1/func/{sub}_ses-1_task-BOLDREST1_dir-AP_run-1_echo-2_part-mag_bold.nii"
    local_file = f"/content/ds006072_work/raw/{sub}.nii"

    cmd = f"aws s3 cp --no-sign-request s3://openneuro.org/ds006072/{remote_file} {local_file}"
    os.system(cmd)

    ts = masker.fit_transform(local_file)
    mat = corr.fit_transform([ts])[0]

    out_csv = f"/content/ds006072_work/matrices/{sub}_connectivity.csv"
    pd.DataFrame(mat).to_csv(out_csv, index=False)

    os.remove(local_file)

    return mat

In [7]:
results = {}

for sub in subjects:
    try:
        print("Processing", sub)
        results[sub] = process_subject(sub)
    except Exception as e:
        print(sub, "failed:", e)

Processing sub-P1


/tmp/ipykernel_2238/2564860997.py:8: FutureWarning: The 'zscore' strategy incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the 'zscore' option will be removed and using standardize=True will fall back to 'zscore_sample'.To avoid this warning, please use 'zscore_sample' instead.
  ts = masker.fit_transform(local_file)
/tmp/ipykernel_2238/2564860997.py:9: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore' option will be removed. and using standardize=True will fall back to 'zscore_sample'.To avoid this warning, please use 'zscore_sample' instead.
  mat = corr.fit_transform([ts])[0]


Processing sub-P2


/tmp/ipykernel_2238/2564860997.py:8: UserWarning: After resampling the label image to the data image, the following labels were removed: {np.float32(64.0), np.float32(65.0), np.float32(71.0), np.float32(73.0), np.float32(14.0), np.float32(20.0), np.float32(21.0), np.float32(23.0), np.float32(30.0)}. Label image only contains 92 labels (including background).
  ts = masker.fit_transform(local_file)
/tmp/ipykernel_2238/2564860997.py:8: FutureWarning: The 'zscore' strategy incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the 'zscore' option will be removed and using standardize=True will fall back to 'zscore_sample'.To avoid this warning, please use 'zscore_sample' instead.
  ts = masker.fit_transform(local_file)
/tmp/ipykernel_2238/2564860997.py:9: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample

Processing sub-P3


/tmp/ipykernel_2238/2564860997.py:8: UserWarning: After resampling the label image to the data image, the following labels were removed: {np.float32(14.0), np.float32(15.0), np.float32(19.0), np.float32(20.0), np.float32(21.0), np.float32(23.0), np.float32(29.0), np.float32(30.0), np.float32(34.0), np.float32(47.0), np.float32(48.0), np.float32(63.0), np.float32(64.0), np.float32(65.0), np.float32(66.0), np.float32(69.0), np.float32(71.0), np.float32(73.0), np.float32(77.0)}. Label image only contains 82 labels (including background).
  ts = masker.fit_transform(local_file)
/tmp/ipykernel_2238/2564860997.py:8: FutureWarning: The 'zscore' strategy incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the 'zscore' option will be removed and using standardize=True will fall back to 'zscore_sample'.To avoid this warning, please use 'zscore_sample' instead.
  ts = masker.fit_transform(

In [8]:
for sub, mat in results.items():
    print(sub, mat.shape)

sub-P1 (100, 100)
sub-P2 (91, 91)
sub-P3 (81, 81)


In [9]:
print("Notebook 2 complete.")
print("Connectivity matrices saved:", len(results))

Notebook 2 complete.
Connectivity matrices saved: 3
